In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, f1_score
from collections import Counter
from tqdm.auto import tqdm
import re

# =========================================================
# 1. CONFIG
# =========================================================

TRAIN_PATH = "TRAIN_PATH"
VAL_PATH   = "VAL_PATH"
TEST_PATH  = "TEST_PATH"

MAX_LEN = 128
BATCH_SIZE = 64
EMBED_DIM = 300

FILTER_SIZES = [2, 3, 4]
NUM_FILTERS = 100

DROPOUT = 0.5

EPOCHS = 10
LR = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)


# =========================================================
# 2. LOAD DATA
# =========================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))


# =========================================================
# 3. CONVERT 3 CLASS → BINARY
# =========================================================

def convert_binary(label):
    if label == 0:
        return 0       # Clean
    else:
        return 1       # Offensive + Hate = Toxic


train_df["binary_label"] = train_df["labels"].apply(convert_binary)
val_df["binary_label"]   = val_df["labels"].apply(convert_binary)
test_df["binary_label"]  = test_df["labels"].apply(convert_binary)


print("\nTRAIN:")
print(train_df["binary_label"].value_counts())

print("\nVAL:")
print(val_df["binary_label"].value_counts())

print("\nTEST:")
print(test_df["binary_label"].value_counts())


# =========================================================
# 4. TOKENIZER ĐƠN GIẢN
# =========================================================

def tokenize(text):
    text = str(text).lower()

    # giữ chữ, số và ký tự tiếng Việt
    tokens = re.findall(r"\w+", text, flags=re.UNICODE)

    return tokens


# =========================================================
# 5. BUILD VOCAB
# =========================================================

counter = Counter()

for text in train_df["texts"]:
    counter.update(tokenize(text))


PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

word2idx = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for word, count in counter.items():
    word2idx[word] = len(word2idx)

vocab_size = len(word2idx)

print("\nVocab size:", vocab_size)


# =========================================================
# 6. ENCODE TEXT
# =========================================================

def encode_text(text):

    tokens = tokenize(text)

    ids = [
        word2idx.get(token, word2idx[UNK_TOKEN])
        for token in tokens
    ]

    # truncate
    ids = ids[:MAX_LEN]

    # padding
    if len(ids) < MAX_LEN:
        ids += [word2idx[PAD_TOKEN]] * (MAX_LEN - len(ids))

    return ids


# =========================================================
# 7. DATASET
# =========================================================

class TextDataset(Dataset):

    def __init__(self, dataframe):

        self.texts = dataframe["texts"].astype(str).tolist()
        self.labels = dataframe["binary_label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        x = torch.tensor(
            encode_text(self.texts[idx]),
            dtype=torch.long
        )

        y = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return x, y


train_dataset = TextDataset(train_df)
val_dataset   = TextDataset(val_df)
test_dataset  = TextDataset(test_df)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# =========================================================
# 8. TEXT CNN
# =========================================================

class TextCNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_classes,
        filter_sizes,
        num_filters,
        dropout
    ):

        super().__init__()

        # Embedding
        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        # CNN filters
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embed_dim,
                out_channels=num_filters,
                kernel_size=filter_size
            )
            for filter_size in filter_sizes
        ])

        # classifier
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            len(filter_sizes) * num_filters,
            num_classes
        )

    def forward(self, x):

        # x:
        # [batch, sequence_length]

        x = self.embedding(x)

        # [batch, sequence_length, embed_dim]

        x = x.permute(0, 2, 1)

        # [batch, embed_dim, sequence_length]

        conv_outputs = []

        for conv in self.convs:

            # [batch, num_filters, new_sequence_length]
            c = torch.relu(conv(x))

            # Global max pooling
            # [batch, num_filters]
            p = torch.max(c, dim=2)[0]

            conv_outputs.append(p)

        # concatenate
        x = torch.cat(conv_outputs, dim=1)

        # [batch, 300]
        # because:
        # 3 filter sizes × 100 filters

        x = self.dropout(x)

        logits = self.fc(x)

        return logits


# =========================================================
# 9. CREATE MODEL
# =========================================================

model = TextCNN(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    num_classes=2,
    filter_sizes=FILTER_SIZES,
    num_filters=NUM_FILTERS,
    dropout=DROPOUT
)

model = model.to(DEVICE)

print("\nModel:")
print(model)

print(
    "\nParameters:",
    sum(p.numel() for p in model.parameters())
)


# =========================================================
# 10. LOSS + OPTIMIZER
# =========================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)


# =========================================================
# 11. TRAIN FUNCTION
# =========================================================

def train_one_epoch(model, loader):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    progress = tqdm(loader, desc="Train")

    for x, y in progress:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(logits, dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

        progress.set_postfix(
            loss=loss.item(),
            acc=correct / total
        )

    return total_loss / len(loader), correct / total


# =========================================================
# 12. VALIDATION
# =========================================================

def evaluate(model, loader):

    model.eval()

    all_preds = []
    all_labels = []

    total_loss = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                y.cpu().numpy()
            )

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    toxic_f1 = f1_score(
        all_labels,
        all_preds,
        pos_label=1
    )

    return (
        total_loss / len(loader),
        acc,
        macro_f1,
        toxic_f1,
        all_labels,
        all_preds
    )


# =========================================================
# 13. TRAIN
# =========================================================

best_f1 = 0

for epoch in range(EPOCHS):

    print(f"\n========== Epoch {epoch+1}/{EPOCHS} ==========")

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    (
        val_loss,
        val_acc,
        val_macro_f1,
        val_toxic_f1,
        _,
        _
    ) = evaluate(
        model,
        val_loader
    )

    print(
        f"\nTrain Loss: {train_loss:.4f}"
    )

    print(
        f"Train Acc:  {train_acc:.4f}"
    )

    print(
        f"Val Loss:   {val_loss:.4f}"
    )

    print(
        f"Val Acc:    {val_acc:.4f}"
    )

    print(
        f"Val Macro-F1: {val_macro_f1:.4f}"
    )

    print(
        f"Val Toxic-F1: {val_toxic_f1:.4f}"
    )

    # save best model
    if val_macro_f1 > best_f1:

        best_f1 = val_macro_f1

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "word2idx": word2idx,
                "config": {
                    "max_len": MAX_LEN,
                    "embed_dim": EMBED_DIM,
                    "filter_sizes": FILTER_SIZES,
                    "num_filters": NUM_FILTERS
                }
            },
            "textcnn_binary_best.pt"
        )

        print(">>> Saved best model")


# =========================================================
# 14. TEST
# =========================================================

checkpoint = torch.load(
    "textcnn_binary_best.pt",
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

(
    test_loss,
    test_acc,
    test_macro_f1,
    test_toxic_f1,
    y_true,
    y_pred
) = evaluate(
    model,
    test_loader
)


print("\n==============================")
print("FINAL TEST RESULT")
print("==============================")

print(f"Accuracy:    {test_acc:.4f}")
print(f"Macro-F1:    {test_macro_f1:.4f}")
print(f"Toxic F1:    {test_toxic_f1:.4f}")

print("\nClassification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Clean",
            "Toxic"
        ],
        digits=4
    )
)

Device: cpu
Train: 14025
Val: 3078
Test: 7357

TRAIN:
binary_label
0    8000
1    6025
Name: count, dtype: int64

VAL:
binary_label
0    2170
1     908
Name: count, dtype: int64

TEST:
binary_label
0    5459
1    1898
Name: count, dtype: int64

Vocab size: 8149

Model:
TextCNN(
  (embedding): Embedding(8149, 300, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(300, 100, kernel_size=(2,), stride=(1,))
    (1): Conv1d(300, 100, kernel_size=(3,), stride=(1,))
    (2): Conv1d(300, 100, kernel_size=(4,), stride=(1,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=300, out_features=2, bias=True)
)

Parameters: 2715602

========== Epoch 1/10 ==========


Train:   0%|          | 0/220 [00:00<?, ?it/s]


Train Loss: 0.5338
Train Acc:  0.7305
Val Loss:   0.4552
Val Acc:    0.7914
Val Macro-F1: 0.7725
Val Toxic-F1: 0.7068
>>> Saved best model

========== Epoch 2/10 ==========


Train:   0%|          | 0/220 [00:00<?, ?it/s]


Train Loss: 0.3558
Train Acc:  0.8495
Val Loss:   0.3775
Val Acc:    0.8343
Val Macro-F1: 0.8083
Val Toxic-F1: 0.7377
>>> Saved best model

========== Epoch 3/10 ==========


...


Train Loss: 0.1047
Train Acc:  0.9618
Val Loss:   0.5152
Val Acc:    0.8522
Val Macro-F1: 0.8276
Val Toxic-F1: 0.7624

========== Epoch 9/10 ==========


Train:   0%|          | 0/220 [00:00<?, ?it/s]


Train Loss: 0.0867
Train Acc:  0.9680
Val Loss:   0.6819
Val Acc:    0.8281
Val Macro-F1: 0.8100
Val Toxic-F1: 0.7513

========== Epoch 10/10 ==========


Train:   0%|          | 0/220 [00:00<?, ?it/s]


Train Loss: 0.0799
Train Acc:  0.9711
Val Loss:   0.5971
Val Acc:    0.8463
Val Macro-F1: 0.8242
Val Toxic-F1: 0.7617

FINAL TEST RESULT
Accuracy:    0.8624
Macro-F1:    0.8269
Toxic F1:    0.7485

Classification Report:
              precision    recall  f1-score   support

       Clean     0.9251    0.8864    0.9053      5459
       Toxic     0.7084    0.7935    0.7485      1898

    accuracy                         0.8624      7357
   macro avg     0.8167    0.8399    0.8269      7357
weighted avg     0.8692    0.8624    0.8649      7357

